In [5]:
# Install libraries
%pip install pandas beautifulsoup4 requests 
%pip install langchain sentence-transformers 
%pip install faiss-cpu streamlit
%pip install -U langchain langchain-community langchain-core langchain-text-splitters
%pip install -U langchain-mistralai  # wrapper (adapteur) officiel Mistral pour LangChain
%pip install -U sentence-transformers faiss-cpu pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
"""
# Config LLM 
mistral_api_key = "nvCuV5PdILu0jQzKcCFnmRTZvYuKSmgB"

"""
# Config LLM
from dotenv import dotenv_values

env_values = dotenv_values("./app.env")

mistral_api_key = env_values.get("MISTRAL_API_KEY", None)

print(mistral_api_key)

""" 
ou mistral_api_key = env_values.get("MISTRAL_API_KEY", None) 
None : valeur par défaut si the key n'existe pas 

"""

nvCuV5PdILu0jQzKcCFnmRTZvYuKSmgB


' \nou mistral_api_key = env_values.get("MISTRAL_API_KEY", None) \nNone : valeur par défaut si the key n\'existe pas \n\n'

In [ ]:
# Instanciation
from langchain_mistralai import ChatMistralAI # ou langchain_mistralai.ChatModels

llm = ChatMistralAI(mistral_api_key=mistral_api_key, model = "mistral-small", temperature=0.5)


In [6]:
# Test the LLM ( Simple )

"""
print(llm ( prompt ) ) : We work with a ChatModel not a text model (llm)

"""

prompt = """Give me 5 moroccan cities""".strip()
  # strip() to remove leading and trailing whitespace
answer = llm.invoke(prompt)

print(answer.content)

1. Marrakech: Known for its bustling Jemaa el-Fnaa square, vibrant markets, and historic sites such as the Koutoubia Mosque and Bahia Palace.

2. Fez: Home to the world's largest medieval city, Fes el-Bali, and its famous tanneries, as well as the University of Al Quaraouiyine, one of the oldest universities in the world.

3. Casablanca: The largest city in Morocco and a major commercial hub, known for its modern architecture, including the Hassan II Mosque, one of the largest mosques in the world.

4. Chefchaouen: A picturesque town in the Rif Mountains, known for its blue-painted streets and whitewashed buildings, as well as its traditional crafts and local cuisine.

5. Rabat: The capital city of Morocco, known for its historic sites such as the Kasbah of the Udayas, the Hassan Tower, and the Royal Palace, as well as its beautiful beaches and gardens.


In [7]:
# Test the LLM (More specific)

response = llm.invoke("Give me 5 Moroccan cities in north morocco")
print(response.content)


1. Tangier: A major city in north Morocco, known for its bustling medina, modern city center, and its location as a port city at the entrance to the Strait of Gibraltar.

2. Tetouan: Also known as the "White Dove," Tetouan is a city in northern Morocco with a rich cultural heritage, located near the Rif Mountains. It's known for its well-preserved medina and its proximity to the blue city of Chefchaouen.

3. Chefchaouen: A beautiful city in the Rif Mountains, known for its blue-painted streets and buildings, making it a popular destination for tourists.

4. Oujda: A city located in eastern Morocco, near the Algerian border, known for its vibrant marketplaces, historic sites, and its location as a gateway to the Sahara Desert.

5. Nador: A coastal city in northeastern Morocco, known for its fishing industry, its location near the Spanish enclave of Melilla, and its proximity to the beautiful blue city of Chefchaouen.


In [17]:
# CSV files >> Langchain Documents
""" 
On ne va pas travailler sur loading du texte brut mais 
sur des documents structurés , donc ca sera tres utile pour detecter 
d'où provient la réponse du chatbot , car chaque LangChain Document a :
page content + metadata (source, date, etc) 

"""
import pandas as pd
from pathlib import Path
from langchain.schema import Document

## Define paths to CSV files
DATA_DIR = Path("C:/Users/info/OneDrive/Desktop/Chatbot - CDG/data/files")  
csv_files = {
    "liens": DATA_DIR/"liens.csv",
    "centres": DATA_DIR/"centres.csv",
    "contacts": DATA_DIR/"contacts.csv",
    "rules": DATA_DIR/"rules.csv",
    "actualites": DATA_DIR/"actualites.csv",
    "RCAR": DATA_DIR/"RCAR.csv"
}


## Function to convert CSV to LangChain Documents
def csv_to_documents(path: Path, source_name: str):
    try:
        df = pd.read_csv(path, sep=";")
    except Exception as e:
        print(f"Erreur lors de la lecture du fichier : {path}")
        print(e)
        return []
    docs = []
    for i, row in df.iterrows():
        text = "\n".join([f"{col}: {row[col]}" for col in df.columns if pd.notna(row[col])])
        docs.append(
            Document(
                page_content=text,
                metadata={"source": source_name, "row_id": i},
            )
        )
    return docs

## Load all documents from CSV files + show result
all_docs = []
for name, path in csv_files.items():
    if path.exists():
        docs = csv_to_documents(path, name)
        if docs:
            print(f"{name}: {len(docs)} documents chargés avec succès.")
        else:
            print(f"{name}: Aucun document chargé (erreur ou fichier vide).")
        all_docs.extend(docs)
    else:
        print(f"{name}: Fichier non trouvé.")

print(f"Total documents chargés: {len(all_docs)}")

liens: 3 documents chargés avec succès.
centres: 4 documents chargés avec succès.
contacts: 1 documents chargés avec succès.
rules: 10 documents chargés avec succès.
actualites: 70 documents chargés avec succès.
RCAR: 1 documents chargés avec succès.
Total documents chargés: 89


In [ ]:
# Chunking documents 
""" 
from langchain.text_splitter import RecursiveCharacterTextSplitter

chunking = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunks = splitter.split_text(texte)

print(f"Nombre de chunks générés : {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\n{chunk}\n")

# encore une fois on ne travail pas sur le texte brut mais 
sur des documents structurés , donc on est besoin d'une autre 
méthode à ajouter pour faire le chunking des Langchain documents

"""

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Choisis la taille des chunks (chunk_size) et le chevauchement (chunk_overlap)
chunking = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# Applique le splitter sur tes documents
chunked_docs = chunking.split_documents(all_docs)

print(f"Total chunks générés : {len(chunked_docs)}")


Total chunks générés : 161


In [ ]:
# Create VectorDB with FAISS
from langchain_community.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings

## Instanciation of systeme embeddings 
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

## Create the VectorDB FAISS à partir des chunks
vector_db = FAISS.from_documents(chunked_docs, embeddings)
print("Vector DB créée avec succès !")
print(f"Nombre de vecteurs : {vector_db.index.ntotal}")

In [39]:
# Define the retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

In [40]:
# Test the retriever ( Simple )
query = "adresse du centre Casablanca"
results = retriever.get_relevant_documents(query)

print(f"Nombre de documents pertinents trouvés : {len(results)}")
for i, doc in enumerate(results):
    print(f"\nDocument {i+1} (source: {doc.metadata.get('source')}):\n{doc.page_content}\n")

Nombre de documents pertinents trouvés : 5

Document 1 (source: centres):
ville,adresse,ouverture,fermeture,type: Casablanca,"CFC Casablanca Finance City Boulevard Main Street, RDC- 60 Tour Main Street",8H30,16H30,centre régional


Document 2 (source: centres):
ville,adresse,ouverture,fermeture,type: Jerrada,"Espace CNRA/RCAR - Société Charbonnage du Maroc, Avenue Hassan II, Quartier Ibn Rouchd.",8H30,16H30,centre régional


Document 3 (source: actualites):
id: 2
titre: CHANGEMENTS D’ADRESSES DES ESPACES CLIENTÈLE
contenu: Le RCAR a le plaisir de vous informer du changement d’adresse de ses espaces clientèle : Casablanca À compter du 14 juillet 2025 , l’espace clientèle est installé au : CFC (Casablanca Finance City) Boulevard Main Street, RDC- 60 Tour Main Street Rabat À compter du 21 juillet 2025 , l’espace clientèle est installé à : Espace des Patios Bâtiment 1, Angle Avenue Annakhil et Mehdi Benbarka Hay Riad- Rabat Au besoin et pour toute information complémentaire, nous vous invi

In [37]:
# Test the retriever (More specific)
query = "le délai de traitement du dossier au mois juillet"
results = retriever.get_relevant_documents(query)

print(f"Nombre de documents pertinents trouvés : {len(results)}")
for i, doc in enumerate(results):
    print(f"\nDocument {i+1} (source: {doc.metadata.get('source')}):\n{doc.page_content}\n")

Nombre de documents pertinents trouvés : 5

Document 1 (source: rules):
id,categorie,description,delai,delai_type,conditions,produit: 5,Paiement Pension,Vous  recevez  le  paiement  de  tout  nouveau  dossier  complet  de  pension Décès ou Invalidité déposé avant le 10 du mois « M » au plus tard le 10 du mois suivant « M+1 ».,,,dossier complet,


Document 2 (source: rules):
id,categorie,description,delai,delai_type,conditions,produit: 10,Autre,Vous recevez au plus tard tous les 27 du mois le paiement mensuel de votre pension « RECORE »,27 du mois,date fixe,,


Document 3 (source: rules):
id,categorie,description,delai,delai_type,conditions,produit: 1,Autre,Vous êtes accueillis avec courtoisie et bienveillance,,,,


Document 4 (source: rules):
id,categorie,description,delai,delai_type,conditions,produit: 2,Autre,Nous sommes joignables tous les jours ouvrés de  08h00 à 18h00  par téléphone et de 08h30 à 16h30 au niveau de nos agences. Nous traitons également vos requêtes en ligne.,,,,




In [24]:
# Test the chatbot (Mistral + RAG)
## Prompt de l'utilisateur 
prompt = "le délai de validation du dossier de décés"

## Recherche des relevant docs dans la VectorDB 
relevant_docs = retriever.get_relevant_documents(prompt)

## Construction du contexte
context = "\n\n".join([doc.page_content for doc in relevant_docs])

## final_prompt = context + prompt
final_prompt = f"Voici des informations extraites de la base de données :\n{context}\n\nQuestion : {prompt}\nRéponds en t'appuyant sur le contexte ci-dessus."

## Génération de réponse par le llm
response = llm.invoke(final_prompt)
print(response.content)

Based on the information provided in the database, the delai (delay) for processing a death pension dossier is specified as follows:

* delai: (blank)
* delai\_type: mois (months)
* conditions: dossier complet

The description states that for a complete death pension dossier received before the 10th of the month "M", the payment will be received by the 10th of the following month "M+1". However, the delay information does not specify the exact number of days or the type of delay for processing the dossier.

Therefore, based on the given context, it can be inferred that the delay for validation of a death pension dossier is within one month from the date of receipt of a complete dossier, but the exact number of days is not specified.


In [25]:
# Test the chatbot (Mistral + RAG)
## Prompt de l'utilisateur 
prompt = "le délai de validation du dossier de décés"

## Recherche des relevant docs dans la VectorDB 
relevant_docs = retriever.get_relevant_documents(prompt)

## Construction du contexte
context = "\n\n".join([doc.page_content for doc in relevant_docs])

## final_prompt = context + prompt
final_prompt = f"Voici des informations extraites de la base de données :\n{context}\n\nQuestion : {prompt}\nRéponds en t'appuyant sur le contexte ci-dessus, réponds comme un assistant travaillant à la rcar"

## Génération de réponse par le llm
response = llm.invoke(final_prompt)
print(response.content)

Le délai de validation d'un dossier de décès à la RCAR est de recevoir le paiement du nouveau dossier de pension de décès ou invalidité déposé complet avant le 10 du mois « M » au plus tard le 10 du mois suivant « M+1 ». Cela signifie que si vous déposez un dossier de pension de décès complet avant le 10 du mois en cours, vous pouvez vous attendre à recevoir le paiement au plus tard le 10 du mois suivant.


In [32]:
# Test the chatbot (Mistral + RAG)
## Prompt de l'utilisateur 
prompt = "C'est quoi la rcar ? "

## Recherche des relevant docs dans la VectorDB 
relevant_docs = retriever.get_relevant_documents(prompt)

## Construction du contexte
context = "\n\n".join([doc.page_content for doc in relevant_docs])

## final_prompt = context + prompt
final_prompt = f"Voici des informations extraites de la base de données :\n{context}\n\nQuestion : {prompt}\nRéponds comme un assistant travaillant à la rcar"

## Génération de réponse par le llm
response = llm.invoke(final_prompt)
print(response.content)

Je suis heureux de vous aider ! Le RCAR, ou Régime Collectif d'Allocation de Retraite, est un régime de retraite marocain qui offre des prestations de retraite, d'invalidité et de décès aux salariés du secteur public et privé, ainsi qu'à leurs ayants droit. Les bénéficiaires du RCAR comprennent les salariés des établissements publics et des collectivités territoriales, ainsi que ceux des entreprises privées qui ont adhéré au régime.

Le RCAR est géré par la Caisse Marocaine des Retraites (CMR), qui est une institution publique sous la tutelle du Ministère de l'Economie, des Finances et de la Réforme de l'Administration. Le RCAR a été créé en 1972 dans le but de fournir une couverture de retraite adéquate et équitable aux salariés marocains.

En tant qu'assistant travaillant pour le RCAR, je suis fier de faire partie d'une organisation qui s'engage à offrir des prestations de retraite de haute qualité à ses bénéficiaires. Nous organisons régulièrement des événements et des rencontres po